# Deploy google/gemma-4-31B-it to EC2 with vLLM and benchmark across instance families

This notebook provisions Amazon EC2 instances, serves
[Gemma 4 31B Instruct](https://huggingface.co/google/gemma-4-31B-it) with vLLM in a
Docker container, then benchmarks each deployment with
[LLMeter](https://github.com/awslabs/llmeter). At the end it
compares throughput and cost across experiments.

**Task under test:** extract a structured JSON booking record from each email. Inputs are loaded from
`sample-data/travel/01-domestic-flight.jsonl` (synthesized by
`sample-data/scripts/synthesize.py --domain travel`).

> **Notes**
> * Each row represents the **optimum packing** for Gemma 4 31B Instruct on that instance — maximum model replicas per instance-hour.
> * Gemma-4-31B (BF16) ~ 62 GiB. Fits on 4× A10G with TP=4 (tight) or 2× L40S (TP=2).
> * Dense 31B Apache-2.0 (released April 2026, ungated).
> * **Default region is us-west-2 (PDX).** Alternates: `us-east-2` (CMH) and `us-east-1` (IAD).

### Prerequisites

1. **AWS credentials** configured for the target account (profile `default` by default). The identity needs EC2 + IAM + SSM permissions.
2. **A default VPC** in the target region.

Everything else — IAM role & instance profile, per-experiment
security group, subnet discovery, AMI lookup, vLLM Docker setup,
and teardown — is handled by `DeploymentRunner`.


## 0. Preparation

### 0.1 Install / verify Python dependencies

In [ ]:
%pip install -q -U \
    "boto3>=1.34" "botocore>=1.34" \
    "llmeter>=0.1.11" "openai>=1.50" \
    "plotly>=5.24" "ipywidgets>=8.1" \
    "huggingface_hub>=0.26" "pandas>=2.2" "matplotlib>=3.9" \
    "requests>=2.32" "tenacity>=9.0" "python-dotenv>=1.0" "jmespath>=1.0"


### 0.2 Imports and logging

In [ ]:
import asyncio
import json
import logging
import os
import sys
from datetime import datetime
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "models" / "gemma_4_31b").is_dir():
    PROJECT_ROOT = NOTEBOOK_DIR
elif NOTEBOOK_DIR.name == "gemma_4_31b":
    PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
else:
    raise RuntimeError(
        f"Can't locate project root from CWD={NOTEBOOK_DIR}. "
        "Launch Jupyter from the project root or from models/gemma_4_31b/."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
_SRC = PROJECT_ROOT / "src"
if _SRC.is_dir() and str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import boto3
import pandas as pd

from vllm_ec2_bench import (
    DeploymentRunner,
    ExperimentConfig,
    catalog_meta,
    upsert_hf_token,
)
from vllm_ec2_bench.cleanup import (
    terminate_all_tagged_instances,
    cleanup_tagged_security_groups,
)
from vllm_ec2_bench.endpoint import VLLMEndpoint
from models.gemma_4_31b import (
    GEMMA_4_31B,
    EXPERIMENTS,
    INSTANCE_TYPES,
    CATALOG_CACHE,
    DEFAULT_REGIONS,
    SYSTEM_PROMPT,
    SEED_INPUT,
    development_experiments,
    get as get_experiment,
    load_catalog,
    refresh_catalog,
)

from llmeter.experiments import LoadTest

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)
LOG = logging.getLogger("notebook")
LOG.info("boto3=%s, pandas=%s", boto3.__version__, pd.__version__)


### 0.3 Region + Hugging Face token configuration

* `REGION` — default region for every experiment (`us-west-2`, PDX).
* `ALT_REGION_1` — first fallback (`us-east-2`, CMH).
* `ALT_REGION_2` — second fallback (`us-east-1`, IAD).
* No HF token required: `google/gemma-4-31B-it` is ungated.


In [ ]:
REGION = "us-west-2"
ALT_REGION_1 = "us-east-2"
ALT_REGION_2 = "us-east-1"

HF_TOKEN = "PLACEHOLDER_PASTE_YOUR_HF_TOKEN"
HF_SECRET_NAME = f"{GEMMA_4_31B.resource_prefix}-benchmark/hf-token"

N_BENCHMARK_SAMPLES = 1000
N_WARMUP_SAMPLES = 5
BENCHMARK_SAMPLE_SEED = 42

assert 1 <= N_BENCHMARK_SAMPLES <= 100_000
assert 0 <= N_WARMUP_SAMPLES <= 1000

print(f"REGION              = {REGION}")
print(f"HF_SECRET_NAME      = {HF_SECRET_NAME}")
print(f"N_BENCHMARK_SAMPLES = {N_BENCHMARK_SAMPLES}")
print(f"N_WARMUP_SAMPLES    = {N_WARMUP_SAMPLES}")


### 0.4 Preflight — AWS identity

In [ ]:
sts = boto3.client("sts")
identity = sts.get_caller_identity()
print(f"Account: {identity['Account']}")
print(f"ARN:     {identity['Arn']}")


### 0.5 Refresh the hardware catalog (pricing + specs from AWS APIs)

In [ ]:
CATALOG = load_catalog(offline_ok=False, max_age_hours_prices=24)
_meta = catalog_meta(CATALOG_CACHE)
print(f"Cache:             {CATALOG_CACHE}")
print(f"Catalog entries:   {len(CATALOG.instance_types())}")
print(f"Prices refreshed:  {_meta.get('prices_refreshed_at', '(just now)')}")
for it in INSTANCE_TYPES[:3]:
    hw = CATALOG.hardware(it)
    prices = CATALOG.price_od_all(it)
    if prices:
        region_prices = ", ".join(f"{r}=${p:.4f}" for r, p in sorted(prices.items()))
    else:
        region_prices = "(price unavailable)"
    print(f"  {it:<18} {hw.num_accelerators}× {hw.accelerator_model:<20} {region_prices}")


### 0.6 Load synthesized benchmark data

Benchmark inputs come from `sample-data/travel/01-domestic-flight.jsonl`
(~10,000 synthesized travel booking confirmation emails). We randomly sample
`N_BENCHMARK_SAMPLES` prompts and reuse the same sample across all
experiments + concurrency tiers so the workload is apples-to-apples.


In [ ]:
import random

synth_file = PROJECT_ROOT.parent / "sample-data" / "travel" / "01-domestic-flight.jsonl"
assert synth_file.exists(), (
    f"{synth_file} is missing. Run "
    f"`python sample-data/scripts/synthesize.py --domain travel` first."
)

all_texts: list[str] = []
with synth_file.open() as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            continue
        text = obj.get("text")
        if text:
            all_texts.append(text)

assert len(all_texts) >= N_BENCHMARK_SAMPLES, (
    f"Synthesis pool has only {len(all_texts)} records; "
    f"need at least N_BENCHMARK_SAMPLES={N_BENCHMARK_SAMPLES}"
)

rng = random.Random(BENCHMARK_SAMPLE_SEED)
INPUTS: list[str] = rng.sample(all_texts, N_BENCHMARK_SAMPLES)

print(f"Synthesis pool:    {len(all_texts):,} records from {synth_file.name}")
print(f"Sampled:           {len(INPUTS):,} prompts (seed={BENCHMARK_SAMPLE_SEED})")
print()
print("--- system prompt ---")
print(SYSTEM_PROMPT)
print()
print("--- first input (abbreviated) ---")
print(INPUTS[0][:320] + ("..." if len(INPUTS[0]) > 320 else ""))


In [ ]:
BENCH_TOTAL_REQUESTS_PER_TIER = N_BENCHMARK_SAMPLES
MAX_NEW_TOKENS = 512
CONCURRENCY_TIERS = [1, 10, 30, 50, 100]
print(f"Per-tier request budget: {BENCH_TOTAL_REQUESTS_PER_TIER}")
print(f"Concurrency tiers:       {CONCURRENCY_TIERS}")
print(f"Warmup requests:         {N_WARMUP_SAMPLES} at c=1 (discarded)")


### 0.7 Shared helper to run one experiment end-to-end

In [ ]:
EXPERIMENTS_STATE: dict[str, dict] = {}

OUTPUT_BASE = Path("outputs")
OUTPUT_BASE.mkdir(exist_ok=True)


async def run_experiment(
    exp_id: str,
    *,
    concurrency_tiers: list[int],
    capacity_reservation_id: str | None = None,
    region_override: str | None = None,
) -> dict:
    cfg = get_experiment(exp_id)
    if region_override:
        new_plan = cfg.deployment.model_copy(update={"region": region_override})
        cfg = cfg.model_copy(update={"deployment": new_plan})

    hw = CATALOG.hardware(cfg.deployment.instance_type)
    print(f"[{exp_id}] spec: {cfg.deployment.instance_type} in {cfg.deployment.region} "
          f"(TP={cfg.deployment.tensor_parallel} DP={cfg.deployment.data_parallel})")
    print(f"[{exp_id}] total VRAM = {hw.vram_gib_total:.1f} GiB across "
          f"{hw.num_accelerators} accelerators")

    runner = DeploymentRunner(
        cfg,
        catalog=CATALOG,
        hf_secret_name=None,
        capacity_reservation_id=capacity_reservation_id,
    )
    state = runner.launch()

    endpoint = VLLMEndpoint(
        base_url=state.base_url,
        api_key=state.api_key,
        model_id=cfg.model_spec.served_model_name,
    )

    # Smoke test
    payload = VLLMEndpoint.create_payload(
        SYSTEM_PROMPT, INPUTS[0], max_tokens=MAX_NEW_TOKENS
    )
    smoke = endpoint.invoke(payload)
    print(f"[{exp_id}] smoke: "
          f"input_tokens={smoke.num_tokens_input} "
          f"output_tokens={smoke.num_tokens_output} "
          f"latency_s={smoke.time_to_last_token:.2f}")

    payloads = [
        VLLMEndpoint.create_payload(SYSTEM_PROMPT, x, max_tokens=MAX_NEW_TOKENS)
        for x in INPUTS[:BENCH_TOTAL_REQUESTS_PER_TIER]
    ]

    if N_WARMUP_SAMPLES > 0:
        n_warmup = min(N_WARMUP_SAMPLES, len(payloads))
        print(f"[{exp_id}] warmup: {n_warmup} passes at c=1 (discarded)")
        warmup_payloads = payloads[:n_warmup]
        warmup = LoadTest(
            endpoint=endpoint,
            payload=warmup_payloads,
            sequence_of_clients=[1],
            output_path=str(OUTPUT_BASE / exp_id / "warmup"),
            min_requests_per_run=n_warmup,
            min_requests_per_client=n_warmup,
        )
        await warmup.run()
        print(f"[{exp_id}] warmup done")

    load_test = LoadTest(
        endpoint=endpoint,
        payload=payloads,
        sequence_of_clients=concurrency_tiers,
        output_path=str(OUTPUT_BASE / exp_id / "load_test"),
        min_requests_per_run=BENCH_TOTAL_REQUESTS_PER_TIER,
        min_requests_per_client=max(1, BENCH_TOTAL_REQUESTS_PER_TIER // max(concurrency_tiers)),
    )
    results = await load_test.run()

    EXPERIMENTS_STATE[exp_id] = {
        "spec": cfg,
        "runner": runner,
        "state": state,
        "endpoint": endpoint,
        "load_test": load_test,
        "results": results,
        "concurrency_tiers": concurrency_tiers,
        "completed_at": datetime.utcnow().isoformat(),
    }
    return EXPERIMENTS_STATE[exp_id]


def teardown_experiment(exp_id: str) -> None:
    entry = EXPERIMENTS_STATE.get(exp_id)
    if not entry:
        print(f"[{exp_id}] no state; nothing to tear down.")
        return
    runner: DeploymentRunner = entry["runner"]
    print(f"[{exp_id}] terminating {runner.state.instance_id} in {runner.state.region}...")
    runner.terminate()
    print(f"[{exp_id}] terminated.")


## Experiment 1 — g5.12xlarge (4× A10G / Ampere)

Capacity strategy: **spot (Fleet) → on-demand → auto-ODCR**.
The deployer walks each mode in order; the final mode is recorded
in `state.capacity_mode` and shown in the comparison table.

Concurrency tiers: `CONCURRENCY_TIERS` (defined in section 0).


In [ ]:
cfg = get_experiment("exp_1")
print(cfg.model_dump())
concurrency_tiers = CONCURRENCY_TIERS
concurrency_tiers


In [ ]:
state_exp_1 = await run_experiment(
    "exp_1",
    concurrency_tiers=concurrency_tiers,
)


**Teardown for exp_1** — also deletes Spot Fleet, Launch Template, and any auto-created ODCR.

In [ ]:
# teardown_experiment("exp_1")  # <-- uncomment to tear down this experiment

## Experiment 2 — g6.12xlarge (4× L4 / Ada)

Capacity strategy: **spot (Fleet) → on-demand → auto-ODCR**.
The deployer walks each mode in order; the final mode is recorded
in `state.capacity_mode` and shown in the comparison table.

Concurrency tiers: `CONCURRENCY_TIERS` (defined in section 0).


In [ ]:
cfg = get_experiment("exp_2")
print(cfg.model_dump())
concurrency_tiers = CONCURRENCY_TIERS
concurrency_tiers


In [ ]:
state_exp_2 = await run_experiment(
    "exp_2",
    concurrency_tiers=concurrency_tiers,
)


**Teardown for exp_2** — also deletes Spot Fleet, Launch Template, and any auto-created ODCR.

In [ ]:
# teardown_experiment("exp_2")  # <-- uncomment to tear down this experiment

## Experiment 3 — g6e.12xlarge (4× L40S / Ada)

Capacity strategy: **spot (Fleet) → on-demand → auto-ODCR**.
The deployer walks each mode in order; the final mode is recorded
in `state.capacity_mode` and shown in the comparison table.

Concurrency tiers: `CONCURRENCY_TIERS` (defined in section 0).


In [ ]:
cfg = get_experiment("exp_3")
print(cfg.model_dump())
concurrency_tiers = CONCURRENCY_TIERS
concurrency_tiers


In [ ]:
state_exp_3 = await run_experiment(
    "exp_3",
    concurrency_tiers=concurrency_tiers,
)


**Teardown for exp_3** — also deletes Spot Fleet, Launch Template, and any auto-created ODCR.

In [ ]:
# teardown_experiment("exp_3")  # <-- uncomment to tear down this experiment

## Experiment 4 — g7e.2xlarge (1× Blackwell)

Capacity strategy: **spot (Fleet) → on-demand → auto-ODCR**.
The deployer walks each mode in order; the final mode is recorded
in `state.capacity_mode` and shown in the comparison table.

Concurrency tiers: `CONCURRENCY_TIERS` (defined in section 0).


In [ ]:
cfg = get_experiment("exp_4")
print(cfg.model_dump())
concurrency_tiers = CONCURRENCY_TIERS
concurrency_tiers


In [ ]:
state_exp_4 = await run_experiment(
    "exp_4",
    concurrency_tiers=concurrency_tiers,
)


**Teardown for exp_4** — also deletes Spot Fleet, Launch Template, and any auto-created ODCR.

In [ ]:
# teardown_experiment("exp_4")  # <-- uncomment to tear down this experiment

## Experiment 5 — g7e.12xlarge (2× Blackwell)

Capacity strategy: **spot (Fleet) → on-demand → auto-ODCR**.
The deployer walks each mode in order; the final mode is recorded
in `state.capacity_mode` and shown in the comparison table.

Concurrency tiers: `CONCURRENCY_TIERS` (defined in section 0).


In [ ]:
cfg = get_experiment("exp_5")
print(cfg.model_dump())
concurrency_tiers = CONCURRENCY_TIERS
concurrency_tiers


In [ ]:
state_exp_5 = await run_experiment(
    "exp_5",
    concurrency_tiers=concurrency_tiers,
)


**Teardown for exp_5** — also deletes Spot Fleet, Launch Template, and any auto-created ODCR.

In [ ]:
# teardown_experiment("exp_5")  # <-- uncomment to tear down this experiment

## Experiment 6 — p4d.24xlarge (8× A100 40GB / Ampere)

Capacity strategy: **spot (Fleet) → on-demand → auto-ODCR**.
The deployer walks each mode in order; the final mode is recorded
in `state.capacity_mode` and shown in the comparison table.

Concurrency tiers: `CONCURRENCY_TIERS` (defined in section 0).


In [ ]:
cfg = get_experiment("exp_6")
print(cfg.model_dump())
concurrency_tiers = CONCURRENCY_TIERS
concurrency_tiers


In [ ]:
state_exp_6 = await run_experiment(
    "exp_6",
    concurrency_tiers=concurrency_tiers,
)


**Teardown for exp_6** — also deletes Spot Fleet, Launch Template, and any auto-created ODCR.

In [ ]:
# teardown_experiment("exp_6")  # <-- uncomment to tear down this experiment

## Experiment 7 — p4de.24xlarge (8× A100 80GB / Ampere)

Capacity strategy: **spot (Fleet) → on-demand → auto-ODCR**.
The deployer walks each mode in order; the final mode is recorded
in `state.capacity_mode` and shown in the comparison table.

Concurrency tiers: `CONCURRENCY_TIERS` (defined in section 0).


In [ ]:
cfg = get_experiment("exp_7")
print(cfg.model_dump())
concurrency_tiers = CONCURRENCY_TIERS
concurrency_tiers


In [ ]:
state_exp_7 = await run_experiment(
    "exp_7",
    concurrency_tiers=concurrency_tiers,
)


**Teardown for exp_7** — also deletes Spot Fleet, Launch Template, and any auto-created ODCR.

In [ ]:
# teardown_experiment("exp_7")  # <-- uncomment to tear down this experiment

## Performance and cost analysis

Per-tier throughput, $/1M-tokens, and percentile-latency comparison across experiments.

In [ ]:
STAT_VARIANT = "average"  # "p50" | "p90" | "p99"

def _get_per_tier_stats(entry: dict) -> dict[int, dict]:
    results_obj = entry.get("results")
    if results_obj is None:
        return {}
    results_dict = getattr(results_obj, "results", None) or {}
    per_tier = {}
    for clients, result in results_dict.items():
        stats = getattr(result, "stats", None)
        if stats is None:
            continue
        per_tier[int(clients)] = stats
    return per_tier


def build_comparison_df(stat_variant: str = "average") -> pd.DataFrame:
    rows: list[dict] = []
    all_tiers: set[int] = set()
    for entry in EXPERIMENTS_STATE.values():
        all_tiers.update(entry["concurrency_tiers"])
    all_tiers_sorted = sorted(all_tiers)

    for exp_id, entry in EXPERIMENTS_STATE.items():
        cfg = entry["spec"]
        dep = cfg.deployment
        hw = CATALOG.hardware(dep.instance_type)
        capacity_mode = entry["state"].capacity_mode
        od_price = CATALOG.price_od(dep.instance_type, dep.region)

        actual_hourly = od_price
        price_source = "OD"
        if capacity_mode == "spot":
            actual_hourly = CATALOG.estimated_spot(dep.instance_type, dep.region) or od_price
            price_source = "spot (estimated, 0.7×OD)*"

        row = {
            "Experiment": exp_id,
            "Instance": dep.instance_type,
            "Region": dep.region,
            "GPUs": hw.num_accelerators,
            "GPU Model": hw.accelerator_model,
            "TP": dep.tensor_parallel,
            "DP": dep.data_parallel,
            "Replicas": cfg.model_replicas,
            "$/hr": round(actual_hourly, 4) if actual_hourly else None,
            "$/hr source": price_source,
            "Capacity": capacity_mode,
        }
        per_tier = _get_per_tier_stats(entry)
        for tier in all_tiers_sorted:
            stats = per_tier.get(tier, {})
            in_tpm = stats.get("average_input_tokens_per_minute") or 0
            out_tpm = stats.get("average_output_tokens_per_minute") or 0
            total_tpm = in_tpm + out_tpm
            cost_per_1m = None
            if total_tpm and actual_hourly:
                cost_per_1m = round(actual_hourly / (total_tpm * 60) * 1_000_000, 4)
            row[f"c={tier} tok/min"] = round(total_tpm, 1) if total_tpm else None
            row[f"c={tier} $/1M"] = cost_per_1m
        rows.append(row)
    return pd.DataFrame(rows) if rows else pd.DataFrame()

df_compare = build_comparison_df(stat_variant=STAT_VARIANT)
csv_path = OUTPUT_BASE / "comparison_table.csv"
df_compare.to_csv(csv_path, index=False)
print(f"Comparison table saved -> {csv_path}")
df_compare


## Cleanup

After benchmarking, make sure **every** instance is terminated. The
emergency sweep below terminates any running instance tagged
`Project={GEMMA_4_31B.project_tag_value}` in all 3 regions.


In [ ]:
# for eid in list(EXPERIMENTS_STATE.keys()):
#     try:
#         teardown_experiment(eid)
#     except Exception as e:
#         print(f"[{eid}] teardown error: {e}")


In [ ]:
PROJECT_TAG = GEMMA_4_31B.project_tag_value
for r in [REGION, ALT_REGION_1, ALT_REGION_2]:
    killed = terminate_all_tagged_instances(r, PROJECT_TAG)
    print(f"{r}: terminated {len(killed)} instance(s): {killed}")
    deleted = cleanup_tagged_security_groups(r, PROJECT_TAG)
    print(f"{r}: deleted {len(deleted)} security group(s): {deleted}")
